![Tracing with LangSmith](assets/banner-adv-04.svg)

LangSmith is LangChain's hosted platform for tracing, evaluating and monitoring LLM applications. It works with or without LangChain. Here we trace the same weather agent so you can compare it directly with Langfuse, then send plain OpenTelemetry spans to it.

**Requirements:** a free account at [smith.langchain.com](https://smith.langchain.com) and an API key in the repo `.env`:

```bash
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=ai-observability-lab
```

Without a key every cell still runs; tracing is simply switched off and the read-back cells are skipped.

In [1]:
import os
import warnings

import lab

warnings.filterwarnings("ignore")  # hide library notices

TRACING = bool(os.getenv("LANGSMITH_API_KEY")) and os.getenv("LANGSMITH_TRACING", "").lower() == "true"
PROJECT = os.getenv("LANGSMITH_PROJECT", "ai-observability-lab")
if os.getenv("LANGSMITH_AGENT_ID"):  # agent routing stores traces in "<agent>-<environment>"
    PROJECT = f"{os.environ['LANGSMITH_AGENT_ID']}-{os.getenv('LANGSMITH_AGENT_ENVIRONMENT', 'local')}"
os.environ.setdefault("LANGSMITH_PROJECT", PROJECT)

if TRACING:
    lab.callout("LangSmith tracing on", f"sending to project <b>{PROJECT}</b>")
else:
    lab.callout(
        "LangSmith tracing off",
        "Add LANGSMITH_API_KEY and LANGSMITH_TRACING=true to .env, restart the kernel and run again.",
        tone="amber",
    )

## The LangSmith vocabulary

| LangSmith | Meaning | Langfuse | OpenTelemetry |
|---|---|---|---|
| **Project** | A bucket of traces, usually one per app and environment | Project | `service.name` |
| **Trace** | One end-to-end request | Trace | Trace |
| **Run** | One step; runs nest into a run tree | Observation | Span |
| **Run type** | `llm`, `chain`, `tool`, `retriever`, `embedding`, `prompt`, `parser` | Observation type | `gen_ai.operation.name` |
| **Thread** | Traces from one conversation, linked by `session_id`, `thread_id` or `conversation_id` metadata | Session | |
| **Feedback** | A score or label attached to a run | Score | |
| **Metadata / tags** | Filterable context | Metadata / tags | Attributes |

Beyond tracing, LangSmith adds **Monitoring** dashboards, **Datasets and Experiments** for offline evaluation, **Annotation Queues** for human review, online **evaluators**, **alerts**, and a **Prompt Hub**.

## 1. Wrap the client, decorate the functions

`wrap_openai` turns each OpenAI call into an `llm` run with tokens and cost. `@traceable` turns any function into a run; `run_type` sets how the UI renders it.

In [2]:
from langsmith import traceable
from langsmith.wrappers import wrap_openai
from openai import OpenAI

client = wrap_openai(OpenAI())
MODEL = "gpt-4o-mini"


@traceable(run_type="retriever")
def search_docs(question: str) -> list[dict]:
    # Retriever runs render best when they return documents with page_content
    return [{"page_content": d, "type": "Document"} for d in lab.search_docs_impl(question)]


@traceable(run_type="tool")
def get_weather(city: str) -> dict:
    for attempt in range(3):
        try:
            return lab.get_weather_impl(city)
        except TimeoutError:
            continue
    return lab.get_weather_impl(city, flaky=False)  # last attempt never times out


@traceable(run_type="chain", name="weather-agent", tags=["notebook-04"], metadata={"prompt_version": "v1"})
def answer_question(question: str, city: str) -> str:
    docs = [d["page_content"] for d in search_docs(question)]
    weather = get_weather(city)
    response = client.chat.completions.create(
        model=MODEL, messages=lab.build_messages(question, docs, weather), max_tokens=80
    )
    return response.choices[0].message.content


print(answer_question("What's the weather in Paris, and should I carry an umbrella?", "Paris"))

LangSmith is not tracing this call: A project ('mytest-local') and an address ({'kind': 'AGENT', 'id': 'mytest', 'environment': 'LOCAL'}) are both set in the environment, so neither outranks the other. Set only one there, or set the one you want at a higher-precedence level.


The weather in Paris is 14°C with a 70% chance of rain, so you should carry an umbrella.


## 2. Conversations become threads

LangSmith groups traces into a **thread** when they share a `session_id`, `thread_id` or `conversation_id` in metadata. `langsmith_extra` lets the caller attach metadata at call time without changing the function.

In [3]:
import uuid

thread_id = str(uuid.uuid4())
for question, city in [
    ("Should I carry an umbrella in London today?", "London"),
    ("And do I need a jacket there?", "London"),
]:
    answer = answer_question(
        question, city, langsmith_extra={"metadata": {"thread_id": thread_id, "user_id": "ana@example.com"}}
    )
    print(f"{question}\n   -> {answer}\n")

Should I carry an umbrella in London today?
   -> Yes, you should carry an umbrella in London today, as the chance of rain is 85%.



And do I need a jacket there?
   -> Yes, you need a jacket in London as it is 11 degrees Celsius, which is below 16 degrees.



## 3. Feedback on a run

To attach feedback you need the run id. `get_current_run_tree()` returns the active run inside a traced function; returning its id lets the frontend send a thumbs up or down later.

In [4]:
from langsmith import Client
from langsmith.run_helpers import get_current_run_tree


@traceable(run_type="chain", name="weather-agent-with-id", tags=["notebook-04"])
def answer_with_run_id(question: str, city: str) -> tuple[str, str | None]:
    run = get_current_run_tree()
    return answer_question(question, city), (str(run.id) if run else None)


answer, run_id = answer_with_run_id("Umbrella needed in Tokyo?", "Tokyo")
print(answer)

if TRACING and run_id:
    ls = Client()
    ls.create_feedback(run_id, key="user_feedback", score=1, comment="thumbs up from the UI")
    ls.flush()
    lab.callout("Feedback sent", f"attached to run <code>{run_id}</code>")

No, an umbrella is not needed in Tokyo since the chance of rain is only 20%.


## 4. Read runs back

The SDK client queries runs with filters, which is useful for reports, regression checks and exporting data.

In [5]:
import time

import pandas as pd

if TRACING:
    time.sleep(5)
    runs = list(Client().list_runs(project_name=PROJECT, is_root=True, limit=10))
    df = pd.DataFrame(
        [
            {
                "name": r.name,
                "status": r.status,
                "latency_s": (r.end_time - r.start_time).total_seconds() if r.end_time else None,
                "tokens": r.total_tokens,
                "cost_usd": float(r.total_cost) if r.total_cost is not None else None,
                "tags": ", ".join(r.tags or []),
            }
            for r in runs
        ]
    )
    display(df.style.format({"latency_s": "{:.2f}", "cost_usd": "{:.6f}"}, na_rep=""))
else:
    print("Skipped: LangSmith tracing is off.")

,name,status,latency_s,tokens,cost_usd,tags
0,travel_agent,success,2.16,594,0.000135,agent
1,weather_assistant,success,1.24,38,0.000009,
2,travel_agent,success,2.20,596,0.000136,agent
3,weather_assistant,success,1.47,38,0.000009,
4,travel_agent,success,2.36,598,0.000137,agent
5,weather_assistant,success,1.63,38,0.000009,
6,probe,success,0.89,14,0.000004,
7,g,success,0.03,0,,
8,smoke-test,success,0.09,0,,


## 5. Plain OpenTelemetry into LangSmith

LangSmith accepts OTLP at `https://api.smith.langchain.com/otel` (regional and self-hosted endpoints differ). Authentication is the `x-api-key` header, and `Langsmith-Project` picks the project. The OTLP/HTTP exporter appends `/v1/traces` itself only when configured through environment variables; in code we pass the full path.

In [6]:
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.trace import SpanKind

endpoint = os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com").rstrip("/") + "/otel/v1/traces"
provider = TracerProvider(resource=Resource.create({"service.name": "weather-agent-otel"}))
if TRACING:
    provider.add_span_processor(
        BatchSpanProcessor(
            OTLPSpanExporter(
                endpoint=endpoint,
                headers={"x-api-key": os.environ["LANGSMITH_API_KEY"], "Langsmith-Project": PROJECT},
            )
        )
    )
tracer = provider.get_tracer("weather-agent")
raw_openai = OpenAI()

with tracer.start_as_current_span("invoke_agent weather-agent") as root:
    root.set_attribute("gen_ai.operation.name", "invoke_agent")
    with tracer.start_as_current_span(f"chat {MODEL}", kind=SpanKind.CLIENT) as span:
        span.set_attribute("gen_ai.operation.name", "chat")
        span.set_attribute("gen_ai.provider.name", "openai")
        span.set_attribute("gen_ai.request.model", MODEL)
        resp = raw_openai.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": "One word for rain."}], max_tokens=5
        )
        span.set_attribute("gen_ai.usage.input_tokens", resp.usage.prompt_tokens)
        span.set_attribute("gen_ai.usage.output_tokens", resp.usage.completion_tokens)
provider.force_flush()
print("OTel spans", "sent to " + endpoint if TRACING else "created locally (tracing off)")

OTel spans sent to https://api.smith.langchain.com/otel/v1/traces


For LangChain and LangGraph apps, LangSmith can also *emit* OpenTelemetry, so the same traces reach another backend too:

```bash
LANGSMITH_OTEL_ENABLED=true     # also export via OTel
LANGSMITH_OTEL_ONLY=true        # export only via OTel
```

## Langfuse or LangSmith?

| | Langfuse | LangSmith |
|---|---|---|
| License and hosting | MIT open source; self-host or cloud | Proprietary; cloud, or self-host on Enterprise |
| Instrumentation | OTel-native SDK, `@observe`, OpenAI drop-in, 50+ integrations | `@traceable`, `wrap_openai`, zero-config for LangChain/LangGraph |
| Conversations | Sessions | Threads |
| Evaluation | Scores, LLM-as-judge, datasets, experiments | Feedback, online evaluators, datasets, experiments, annotation queues |
| Accepts OTLP | Yes | Yes |
| Pick it when | You need data control, self-hosting, or open source | Your stack is LangChain/LangGraph, or you want a managed product |

Both are built around the same trace and span model, so the skills transfer. Next: [05 Analytics](05_analytics.ipynb).